# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured walkthrough for loading, exploring, and processing the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and records from the Croissant dataset using `mlcroissant`. The metadata provides context for what record sets and fields are available.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata as a JSON object (dictionary)
metadata = dataset.metadata.to_json()

print("Dataset loaded!")
print("\nTitle:", metadata.get("name", ""))
print("Description:", metadata.get("description", ""))
print("\nKeywords:", metadata.get("keywords", ""))


## 2. Data Overview
Explore the available record sets and their associated fields/columns.

**Note:** All entities are referenced by their `@id`. This ensures you can trace and manipulate data correctly.

In [ ]:
# Display available record sets and their fields using only @id for references
def get_record_sets(ds):
    all_record_sets = ds.metadata.record_sets
    record_sets_list = []
    for rs in all_record_sets:
        # Each record set has an '@id' and 'field' list
        fields = getattr(rs, 'fields', [])
        # Some schemas may call them 'fields' or 'columns', check both
        if not fields and hasattr(rs, 'columns'):
            fields = getattr(rs, 'columns', [])
        record_sets_list.append({'@id': rs.id, 'fields': [f.id for f in fields]})
    return record_sets_list

record_sets = get_record_sets(dataset)
print("Record Sets in dataset (by @id):\n")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. RecordSet @id: {rs['@id']}")
    print(f"   Fields/Columns @id:")
    for f in rs['fields']:
        print(f"       - {f}")
    print()
# Save the record set ids for later
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load records from a selected record set into a pandas DataFrame for analysis. Always reference record sets and fields by their `@id` as displayed above.

In [ ]:
dataframes = dict()

for record_set_id in record_set_ids:
    print(f"Loading RecordSet {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict with field @id as the key
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f" -> {len(records)} records; columns (by field @id): {list(dataframes[record_set_id].columns)}\n")
    else:
        print(" -> No records available in this record set.\n")
# For demonstration, pick the first non-empty record set
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break

if main_record_set_id:
    print(f"Using RecordSet {main_record_set_id} for EDA.")
    print(f"Fields present (by @id): {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No populated record set found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. (Use field `@id` throughout for all references.)

> **Note:** If your dataset and fields differ, adapt the field `@id` accordingly.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Select a numeric field (by @id); example picks the first float/integer column
    sample_numeric_field = None
    for col in df.columns:
        # Try to infer numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            sample_numeric_field = col
            break

    if sample_numeric_field:
        print(f"Using numeric field for analysis: {sample_numeric_field}")
        # Set a threshold (use 75th percentile if unsure)
        threshold = df[sample_numeric_field].quantile(0.75)
        filtered_df = df[df[sample_numeric_field] > threshold].copy()
        print(f"Filtered records with {sample_numeric_field} > {threshold:.2f} (top quartile):")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{sample_numeric_field}_normalized"] = (filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()) / filtered_df[sample_numeric_field].std()
        print(f"Normalized {sample_numeric_field} for filtered records:")
        display(filtered_df[[sample_numeric_field, f"{sample_numeric_field}_normalized"]].head())

        # Try grouping by another field, e.g. first categorical column
        sample_group_field = None
        for col in df.columns:
            if col != sample_numeric_field and df[col].dtype == object:
                sample_group_field = col
                break
        if sample_group_field:
            grouped = filtered_df.groupby(sample_group_field)[sample_numeric_field].mean().to_frame("mean_").reset_index()
            print(f"Grouped data by {sample_group_field} and calculated mean({sample_numeric_field}):")
            display(grouped.head())
        else:
            print("No categorical (object) field found to group by.")
    else:
        print("No numeric field found for analysis in the selected record set.")
else:
    print("No record set to analyze.")

## 5. Visualization
Visualize data distributions or relationships. Make sure you reference the correct field `@id`.

Below, we visualize the filtered numeric field's distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and sample_numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[sample_numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {sample_numeric_field}")
    plt.xlabel(sample_numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if sample_group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=sample_group_field, y=sample_numeric_field, data=filtered_df)
        plt.title(f"{sample_numeric_field} by {sample_group_field}")
        plt.xlabel(sample_group_field)
        plt.ylabel(sample_numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
You have successfully loaded and explored the FAIR<sup>2</sup> dataset using `mlcroissant`.

- You inspected available record sets and fields using their `@id`.
- You extracted records and assembled them in pandas DataFrames.
- Performed typical EDA: filtering on a numeric field, normalization, and grouping by a categorical variable.
- Visualized key properties such as value distributions.

Continue your analysis by adapting the field and record set `@id`s to your specific research goals!